In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/291-rag-dataset/Main/Informatics impact requires effective, scalable tools and standards-based infrastructure.pdf
/kaggle/input/291-rag-dataset/Main/Sharing Biomedical Data_ Strengthening AI Development in Healthcare.pdf
/kaggle/input/291-rag-dataset/Main/The global prevalence of myocardial infarction_ a systematic review and meta-analysis.pdf
/kaggle/input/291-rag-dataset/Main/What is good mental health_ A scoping review.pdf
/kaggle/input/291-rag-dataset/Main/Machine learning for precision medicine1.pdf
/kaggle/input/291-rag-dataset/Main/100 Years of Insulin.pdf
/kaggle/input/291-rag-dataset/Main/Recent Advancements in Pathogenesis, Diagnostics and Treatment of Alzheimer’s Disease.pdf
/kaggle/input/291-rag-dataset/Main/What do people really think of generic medicines_ A systematic review and critical appraisal of literature on stakeholder perceptions of generic drugs.pdf
/kaggle/input/291-rag-dataset/Main/Advancing the National Immunization Program in an era of achieving

## Config

In [3]:
import os

DATA_DIR = "/kaggle/input/291-rag-dataset/Main"   # update to your dataset path
REQUESTS_PATH = "requests.json"
MANUAL_BASELINE_PATH = "manual_baseline.json"
CHUNKS_PATH = "chunks_catalog.json"
EMB_PATH = "chunk_embeddings.npy"
FAISS_INDEX_PATH = "faiss.index"
LLM_PROMPTS_PATH = "llm_prompts_for_calls.json"
RETRIEVAL_RESULTS_PATH = "faiss_results_chunks.json"
EVAL_PATH = "evaluation_chunks.json"

TOP_K = 8
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# chunking
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200

# performance / parallelism
EMBED_BATCH = 128
PDF_LOAD_WORKERS = 6
CHUNK_PROC_WORKERS = 4


## Imports & helpers

In [4]:
!pip install -q PyPDF2 tqdm sentence-transformers numpy faiss-cpu scikit-learn
import time
import json
import uuid
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from typing import List, Dict, Any, Tuple

from tqdm import tqdm
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.metrics import ndcg_score

def now(): return time.time()
def safe_print(*a, **k): print(*a, **k)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

2025-12-05 08:32:04.863571: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764923525.043253      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764923525.111871      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

## Document reader (parallel)

In [5]:
from pathlib import Path

def read_file_text(path: str) -> Dict[str, str]:
    file_name = Path(path).name
    text = ""
    try:
        if path.lower().endswith(".pdf"):
            reader = PdfReader(path)
            pages = []
            for p in reader.pages:
                page_text = p.extract_text() or ""
                pages.append(page_text)
            text = "\n".join(pages)
        else:
            with open(path, "r", errors="ignore") as f:
                text = f.read()
    except Exception as e:
        safe_print(f"[read_file_text] Warning: failed to read {file_name}: {e}")
        text = ""
    return {"id": file_name, "text": text}

def load_documents_parallel(data_dir: str, min_chars: int = 200, max_workers: int = PDF_LOAD_WORKERS) -> List[Dict]:
    files = [str(Path(data_dir)/f) for f in sorted(os.listdir(data_dir))]
    docs = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(read_file_text, fp): fp for fp in files}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Loading documents"):
            res = fut.result()
            if res and len(res["text"].strip()) >= min_chars:
                docs.append(res)
    return docs

safe_print("Sample files:", os.listdir(DATA_DIR)[:5])

Sample files: ['Informatics impact requires effective, scalable tools and standards-based infrastructure.pdf', 'Sharing Biomedical Data_ Strengthening AI Development in Healthcare.pdf', 'The global prevalence of myocardial infarction_ a systematic review and meta-analysis.pdf', 'What is good mental health_ A scoping review.pdf', 'Machine learning for precision medicine1.pdf']


## Chunking

In [6]:
def chunk_text_sliding_window(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if chunk_size <= 0: raise ValueError("chunk_size must be positive")
    if overlap >= chunk_size: raise ValueError("overlap must be smaller than chunk_size")
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap
    return chunks

def chunk_doc_pair(args: Tuple[str, str]) -> List[Dict]:
    doc_id, text = args
    chs = chunk_text_sliding_window(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    out = []
    for i, t in enumerate(chs):
        chunk_id = f"{doc_id}__c{i:04d}"
        out.append({"chunk_id": chunk_id, "doc_id": doc_id, "chunk_index": i, "text": t})
    return out

def build_chunk_catalog_parallel(documents: List[Dict], workers: int = CHUNK_PROC_WORKERS) -> List[Dict]:
    tasks = [(d["id"], d["text"]) for d in documents]
    all_chunks = []
    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(chunk_doc_pair, t) for t in tasks]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Chunking documents"):
            res = fut.result()
            all_chunks.extend(res)
    return all_chunks


## Create chunks

In [7]:
documents = load_documents_parallel(DATA_DIR, min_chars=200, max_workers=PDF_LOAD_WORKERS)
safe_print(f"Loaded {len(documents)} documents. Starting chunking...")
chunks = build_chunk_catalog_parallel(documents, workers=CHUNK_PROC_WORKERS)
safe_print(f"Created {len(chunks)} chunks.")
with open(CHUNKS_PATH, "w") as f:
    json.dump(chunks, f)
safe_print(f"Saved chunk catalog to {CHUNKS_PATH}")


Loading documents:  18%|█▊        | 21/119 [00:06<00:41,  2.34it/s]WARNING:PyPDF2._cmap:unknown widths : 
[0, IndirectObject(10, 0, 135209524592592)]
Loading documents:  18%|█▊        | 22/119 [00:08<00:58,  1.67it/s]WARNING:PyPDF2._cmap:unknown widths : 
[0, IndirectObject(16, 0, 135209524592592)]
[0, IndirectObject(22, 0, 135209524592592)]
Loading documents:  19%|█▉        | 23/119 [00:09<01:08,  1.39it/s]WARNING:PyPDF2._cmap:unknown widths : 
[0, IndirectObject(28, 0, 135209524592592)]
[0, IndirectObject(34, 0, 135209524592592)]
Loading documents:  20%|██        | 24/119 [00:09<00:59,  1.59it/s]WARNING:PyPDF2._cmap:unknown widths : 
[0, IndirectObject(40, 0, 135209524592592)]
[0, IndirectObject(46, 0, 135209524592592)]
[0, IndirectObject(52, 0, 135209524592592)]
Loading documents: 100%|██████████| 119/119 [05:38<00:00,  2.84s/it]

Loaded 119 documents. Starting chunking...



Chunking documents: 100%|██████████| 119/119 [00:00<00:00, 389.87it/s]


Created 27213 chunks.
Saved chunk catalog to chunks_catalog.json


## Build chunk embeddings

In [8]:
def load_or_build_embeddings(chunks: List[Dict], model_name: str = MODEL_NAME, emb_path: str = EMB_PATH, batch_size: int = EMBED_BATCH):
    texts = [c["text"] for c in chunks]
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"
    safe_print(f"Loading embedding model '{model_name}' on device: {device}")
    model = SentenceTransformer(model_name, device=device)
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    embeddings = embeddings.astype("float32")
    np.save(emb_path, embeddings)
    safe_print(f"Saved embeddings to {emb_path}")
    return embeddings

# Run embedding build/load (this may take time on CPU)
if 'chunks' not in globals():
    raise RuntimeError("Chunks not defined. Run Cell 6 first.")
embeddings = load_or_build_embeddings(chunks, model_name=MODEL_NAME, emb_path=EMB_PATH, batch_size=EMBED_BATCH)
safe_print("Embeddings shape:", embeddings.shape)

Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' on device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/213 [00:00<?, ?it/s]

Saved embeddings to chunk_embeddings.npy
Embeddings shape: (27213, 384)


## Build FAISS index

In [9]:
def build_faiss_index(embeddings: np.ndarray, index_path: str = FAISS_INDEX_PATH, nlist_threshold: int = 50000):
    emb = embeddings.copy()
    faiss.normalize_L2(emb)
    dim = emb.shape[1]
    n_vectors = emb.shape[0]

    if n_vectors >= nlist_threshold:
        nlist = max(256, min(4096, n_vectors // 100))
        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        safe_print(f"[faiss] Training IVF index (nlist={nlist}) on {n_vectors} vectors...")
        index.train(emb)
        batch = 20000
        for i in tqdm(range(0, n_vectors, batch), desc="Adding vectors to IVF index"):
            index.add(emb[i:i+batch])
    else:
        index = faiss.IndexFlatIP(dim)
        batch = 20000
        for i in tqdm(range(0, n_vectors, batch), desc="Adding vectors to Flat index"):
            index.add(emb[i:i+batch])

    faiss.write_index(index, index_path)
    safe_print(f"Saved FAISS index to {index_path}")
    return index

def load_faiss_index(index_path: str):
    if os.path.exists(index_path):
        safe_print(f"Loading FAISS index from {index_path}")
        return faiss.read_index(index_path)
    return None

# Execute build    
index = build_faiss_index(embeddings, index_path=FAISS_INDEX_PATH)

Adding vectors to Flat index: 100%|██████████| 2/2 [00:00<00:00, 30.30it/s]

Saved FAISS index to faiss.index


## Load requests and retrieve top-k for all queries

In [10]:
requests = [
"Which groups are considered at highest risk of severe influenza complications, and what medical or preventive interventions are recommended for them?",
"What are the major risk factors and early symptoms of myocardial infarction, and how does public awareness impact treatment outcomes?",
"How can early diagnosis, evolving clinical definitions, and global prevention strategies improve sepsis outcomes and reduce mortality worldwide?",
"Based on the collective perspective of recent literature, what are the major biological, technological, and sociopolitical barriers that still hinder full malaria eradication despite modern advances?",
"What evidence exists across sources that mass vaccination programs have transformed global mortality and morbidity patterns, and what indicators suggest this progress is at risk?",
"How do logistical, social, and structural barriers affect vaccine delivery in rural or underserved regions, and what community-based interventions have shown promise in mitigating these obstacles?",
"Which emerging therapies, including gene therapy and molecular targeted approaches, show promise in revolutionizing overall cancer treatment paradigms?",
"Which innovations in smart healthcare systems are improving disease diagnosis and clinical decision-making?",
"What key social and environmental factors have been identified as influencing mental health outcomes, and how do recent studies suggest addressing them?",
"What role do digital technologies and mobile platforms play in improving diabetes management and patient engagement?",
"What is insulin, how does it regulate blood glucose, and how has insulin therapy evolved over time?"]

with open(REQUESTS_PATH, "w") as f:
    json.dump(requests, f, indent=2)
safe_print("Wrote a sample requests.json (edit it to add your queries)")

def retrieve_chunks_for_queries(requests: List[str], model_name: str, index: faiss.Index, chunks_catalog: List[Dict], top_k: int = TOP_K, batch_size: int = 64):
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"
    model = SentenceTransformer(model_name, device=device)
    q_embs = model.encode(requests, batch_size=batch_size, convert_to_numpy=True, show_progress_bar=True)
    q_embs = q_embs.astype("float32")
    faiss.normalize_L2(q_embs)
    D, I = index.search(q_embs, top_k)
    results = {}
    for qi, q in enumerate(requests):
        ids = []
        docs = []
        for idx in I[qi]:
            if idx < 0 or idx >= len(chunks_catalog):
                continue
            ids.append(chunks_catalog[idx]["chunk_id"])
            docs.append(chunks_catalog[idx]["doc_id"])
        results[q] = {"chunk_ids": ids, "doc_ids": docs}
    return results

# Run retrieval
if 'index' not in globals():
    raise RuntimeError("FAISS index not defined. Run Cell 8 first.")
results_chunks = retrieve_chunks_for_queries(requests, MODEL_NAME, index, chunks, top_k=TOP_K, batch_size=64)
safe_print(f"Retrieved top-{TOP_K} chunks for {len(requests)} queries.")
with open(RETRIEVAL_RESULTS_PATH, "w") as f:
    json.dump(results_chunks, f, indent=2)
safe_print(f"Saved retrieval results to {RETRIEVAL_RESULTS_PATH}")

Wrote a sample requests.json (edit it to add your queries)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved top-8 chunks for 11 queries.
Saved retrieval results to faiss_results_chunks.json


## Build prompts for each query using retrieved chunks

In [11]:
def build_prompt_from_chunks(question: str, retrieved_chunk_ids: List[str], chunks_catalog: List[Dict], max_chunk_chars: int = 1200):
    system = ("You are a medical evidence summarizer. Use ONLY the provided chunks. "
              "Cite chunk_ids for every factual claim. If answer not present, return INSUFFICIENT_DATA.")
    parts = [system, f"\nUser question:\n{question}\n\nRetrieved chunks:\n"]
    for i, cid in enumerate(retrieved_chunk_ids, start=1):
        cobj = next((c for c in chunks_catalog if c["chunk_id"] == cid), None)
        if not cobj:
            continue
        txt = cobj["text"]
        if len(txt) > max_chunk_chars:
            txt = txt[:max_chunk_chars].rsplit(" ", 1)[0] + "..."
        parts.append(f"{i}) [chunk_id: {cobj['chunk_id']}] [doc_id: {cobj['doc_id']}]\n{txt}\n")
    schema = """
Return ONLY a single JSON object EXACTLY matching this schema:
{
  "answer": "<100-150 word answer>",
  "evidence": [
    {"chunk_id": "<chunk_id>", "quote": "<exact quote or short excerpt>", "supporting_text_summary": "<1 sentence>"}
  ],
  "confidence": "low|medium|high",
  "extraction_type": "extractive|synthesized|insufficient",
  "suggested_followups": ["..."]
}
If information is insufficient to answer the question using only the chunks above,
set extraction_type to "insufficient" and answer to "INSUFFICIENT_DATA".
"""
    parts.append(schema)
    return "\n".join(parts)

llm_prompts = {}
for q, res in results_chunks.items():
    prompt = build_prompt_from_chunks(q, res["chunk_ids"], chunks, max_chunk_chars=5000)
    llm_prompts[q] = prompt

with open(LLM_PROMPTS_PATH, "w") as f:
    json.dump(llm_prompts, f)
safe_print(f"Saved {len(llm_prompts)} prompts to {LLM_PROMPTS_PATH}")

Saved 11 prompts to llm_prompts_for_calls.json


## Manual Baseline

In [12]:
manual_baseline = {

    "Based on the collective perspective of recent literature, what are the major biological, technological, and sociopolitical barriers that still hinder full malaria eradication despite modern advances?": [
        "In the third decade of the 21st century, malaria remains a significant challenge to the health of humans living where it is transmitted. The encouraging trends observed in the first years of the millennium are now threatened by a stall in progress reducing malaria cases. The new push for malaria eradication, which monopolized most malaria efforts in the last decade seems to have been tampered by a renewed spirit of enhanced efforts in those countries with higher burdens, recognizing that lack of improvements there will thwart global progress. In this particular moment, innovation and research focussing on new therapeutics, diagnostics and insecticides need to continue at the forefront of global efforts, so as to overcome the biological challenges affecting the tools available, and thus creatively bypass the hurdles that are hampering our fight against this deadly disease.",
        "The endemism of malaria in West Africa, for example, has huge economic impacts. In Ghana, the cost of treating a single episode of malaria can reach 34% of a household’s yearly\n\nincome. As a result of infected people being unable to produce wealth, and as wealth is low, the risk of exposure to malaria increases. There is a negative relationship between national economic growth and malaria control expenditures. As a result, malaria can be considered both as a cause of poverty and as a consequence of it. Furthermore, past interventions have been ineffective as they were unable to account for the nonlinear nature of malaria infection. Particularly, the unexpected consequences of interventions such as mosquito resistance to chemical compounds and parasite resistance to drugs led to their abandonment. This was the case with the eradication control program that was abandoned in 1969 and the Global Malaria Control Strategy in 1992. Scientists have widely acknowledged the complexity of malaria, but a depiction of this complexity is weakly documented, and further limits the scope of malaria control.",
        "The first 15 years of the millennium showed a stable and consistent reduction in the malaria burden, with important changes in its geographic distribution (Figure 1), and an overall reduction of around 60% in terms of malaria mortality. This led to a renewed enthusiasm, endorsed by the Global health community, towards a second push for malaria eradication. During the last 5 years, however, progress in terms of the malaria burden seems to have stalled, with a set of countries adequately progressing towards malaria elimination, but other countries showing an alarming increasing incidence of their malaria burden. P. falciparum accounts for the vast majority of malaria cases in the WHO African region (>99%), but is also a problem in regions of the Western Pacific (71.9%), the Eastern Mediterranean (69%) and South-East Asia (62.8%). P. vivax is the parasite driving the burden in in the Americas and South-east Asia, and, apart from some specific regions of the Horn of Africa, is very uncommon across the rest of Africa due to the absence of Duffy antigen in human populations"
    ],

    "What evidence exists across sources that mass vaccination programs have transformed global mortality and morbidity patterns, and what indicators suggest this progress is at risk?": [
        "Vaccines have transformed public health, particularly since national programmes for immunization first became properly established and coordinated in the 1960s. In countries with high vaccine programme coverage, many of the diseases that were previously respon-sible for the majority of childhood deaths have essentially disappeared 1. The World Health Organization (WHO) estimates that 2-3 million lives are saved each year by current immunization programmes, contributing to the marked reduction in mortality of children less than 5 years of age globally from 93 deaths per 1,000 live births in 1990 to 39 deaths per 1,000 live births in 2018. Vaccines exploit the extraordinary ability of the highly evolved human immune system to respond to, and remember, encounters with pathogen antigens. However, for much of history, vaccines have been developed through empirical research without the involvement of immunologists. There is a great need today for improved understanding of the immunological basis for vaccination to develop vaccines for hard-to-target pathogens (such as Mycobacterium tuberculosis, the bacterium that causes tuberculosis (TB)) and antigenically variable pathogens (such as HIV), to control outbreaks that threaten global health security (such as COVID-19 or Ebola) and to work out how to revive immune responses in the ageing immune system to protect the growing population of older adults from infectious diseases.",
        "Fourth, examination of excess mortality data reveals these deaths are largely not attributed to COVID-19 in official statistics, yet they correlate strongly with vaccination uptake across countries and time periods. When Bradford Hill criteria for causation are applied — including strength of association, consistency across populations, temporality, biological gradient (dose-response), and biological plausibility — the evidence suggests these correlations warrant serious consideration as potentially causal relationships.",
        "The evidence chain — from indistinguishable symptoms between vaccination and COVID, to spike protein circulation, to testing confusion, to excess mortality correlations — suggests the conventional understanding may have gotten the relationship between vaccination and mortality backwards. Rather than preventing deaths, the intervention may have contributed to them through mechanisms the models never considered. As Engler (2022) noted in analyzing the Lombardy mortality spike, synchronous increases in deaths across regions suggested policy responses rather than viral spread as the primary driver.",
        "A striking pattern has emerged in the scientific literature examining COVID-19 vaccine effectiveness and excess mortality: studies based on mathematical modeling consistently show dramatic benefits from vaccination campaigns, while empirical analyses of real-world data often find minimal, negligible, or even negative effects."
    ],

    "How do logistical, social, and structural barriers affect vaccine delivery in rural or underserved regions, and what community-based interventions have shown promise in mitigating these obstacles?": [
        "Low education, low awareness of HPV vaccination, lack of strong provider recommendations and barriers to appropriate healthcare are cited as some of the most prevalent reasons for low uptake and completion in the U.S. Rural settings are particularly vulnerable to these barriers with studies demonstrating that rural populations are less likely to be vaccinated. Rural and urban disparities in HPV vaccination have emerged as a concern, with approximately 60 million Americans (almost one-fifth of the population) living in rural locations. Recent research efforts have sought to determine ways to reduce these barriers and improve HPV vaccination specifically in rural populations. Among the most promising opportunities to increase HPV vaccination rates, particularly in rural settings, is by utilizing pharmacies as alternative vaccination sites. There is a wealth of existing literature advocating for pharmacies to serve in this capacity to reduce barriers and improve access to HPV vaccination, including the 2018 President’s Cancer Panel report on HPV vaccination. Some existing resources unique to many pharmacies making them ideal alternative settings for vaccination include: infrastructure and storage capacity for various medications, established vaccine protocols, longer hours than traditional healthcare provider offices as well as weekend availability, and convenience/greater accessibility. In 2016, the U.S. National Vaccine Program Office reported that 95% of Americans lived within five miles of a community pharmacy, removing a critical access barrier for rural populations that have to travel substantial distances to reach a provider.",
        "Civil unrest limiting vaccine delivery was also identified as a major challenge in achieving rubella elimination goal. There are several strategies that may be used in conflict-affected countries to increase vaccination coverage, including the organization of vaccination campaigns (vaccination catch-up) and the use of outreach services (vaccination of children in remote locations on scheduled dates known by the communities) in collaboration with local communities and the military or security personnel to guarantee a safe passage and security for HCWs. Such a post-conflict catch-up vaccination campaign has been successfully conducted in the Central African Republic. Gavi, the Vaccine Alliance, has requested that governments support the costs of civil society organizations for the development of vaccine stockpiles for humanitarian emergencies, the strengthening of healthcare systems, and the improvement of the access and affordability of vaccines in conflict-affected areas. As social and military crises, public health crises of high magnitude, such as the COVID-19 pandemic, have also the potential to disrupt the delivery of basic healthcare and immunization services, including routine vaccination campaigns. Despite the WHO recommendations to maintain essential health services during outbreaks [98], several countries have recorded significant decreases in vaccination coverage due to the COVID-19 pandemic, leading to increased risk for outbreaks of several vaccine-preventable diseases [99,100]. Disparities in this decline in vaccination coverage were observed, and low- to middle-income countries were particularly affected [46]. The main factors contributing to this decline included the fear of being exposed to the virus at healthcare facilities, restrictions in terms of population movements, short-age of HCWs, and overloaded healthcare systems [46]. Among others, measles vaccination campaigns were paused or postponed in several countries to limit the propagation of COVID-19 [47]. In these regions, it is particularly important to track the children who did not receive measles-containing vaccines and to ensure that they are immunized as quickly as possible when the safety of communities and HCWs can be ensured. In summary, the common goal of all interventions aimed at restoring disrupted vaccination coverage due to the above-mentioned reasons is to ensure that all communities and individuals who were not vaccinated through routine immunization services receive their vaccines. The organization of supplementary immunization activities (SIAs; vaccination campaigns during which the targeted populations are immunized regardless of their vaccination history) and catch-up campaigns, as well as the strengthening of outreach services are major initiatives to reach this objective [1,25,101–104]. In 2017, approximately 205 million doses of measles-containing vac- cine were administered during 53 SIAs in 39 countries with a low vaccination coverage rate and a high measles burden [25,102]. Additional solutions to improve vaccination service arrangements in low- to middle-income countries include home visits and inter- and intra-facility referral linkages, which are particularly important when families move between vaccinations [104].",
        "In 2016, the U.S. National Vaccine Program Office reported that 95% of Americans lived within five miles of a community pharmacy, removing a critical access barrier for rural populations that have to travel substantial distances to reach a provider [13]. Close provider proximity is additionally beneficial for vaccines such as the HPV vaccine which consists of multiple doses, increasing the likelihood that patients will follow-up to complete the series [23]. Community pharmacies (frequently independently owned and operated versus a large chain store) are particularly important for consideration as alternative settings for HPV vaccination. In rural and small-town settings, these pharmacies are frequently an established part of the community. Community pharmacists are often well-known, trusted members of the community who act as first-line healthcare providers [7]. Pilot studies have looked at implementation of interventions offering HPV vaccination in pharmacy settings. In 2018, Michigan researchers implemented a\n\npharmacy-based pilot in 10 retail pharmacies, with the objective of identifying barriers, challenges, and successes with regard to HPV vaccination. Major findings included barriers specific to the HPV vaccine (specific age range, social stigma, the anti-vaccination movement) that were distinct from barriers to the influenza vaccine; low pharmacist confidence in recommending the vaccine; and high cost for patients. At roughly $250 per dose, cost is a significant barrier to overcome without assistance programs [24]. In the most recent study, performed by Calo et al. in 2019, HPV vaccination protocols were implemented in 15 clinics throughout five states. However, only 13 doses of the HPV vaccine were given over a span of 1 year. Researchers identified various explanations for the significantly underperforming results, including low parent demand, low engagement among pharmacy staff, poor-third party reimbursement, and limited integration into primary care systems [25].",
        "To our knowledge, no studies have focused on rural community pharmacies. Rural populations are often medically underserved and overlooked in research [7]. Considering 20% of the American population lives in rural areas, it is imperative that efforts be made to address their significant barriers to healthcare [10, 11]. One barrier is lack of access to care; for example, 22 of 67 counties in Alabama, a largely rural state, lack a pediatrician and 2 counties do not have any primary care providers [26]. Although previous pharmacy studies have focused on increasing access to HPV vaccination, most have not addressed the frequently inextricable economic barriers that many in rural settings face [8]. Individuals in rural areas are more likely to be poor and more likely to rely on Medicaid than their urban counterparts [27, 28]. Children on Medicaid have limited vaccination options because they can only receive immunizations from Vaccines for Children (VFC) providers. The VFC program provides free vaccinations to children who might not otherwise receive the vaccine because of inability to pay [29]. Therefore, increasing access points alone in rural settings does not adequately address the needs of these populations. Additional vaccination settings must be approved VFC providers if they are to effectively serve the needs of their population [8]. The current pilot study examined the feasibility and efficacy potential of enrolling a rural, community pharmacy as a VFC provider to increase access and promotion of HPV vaccination."
    ],

    "Which groups are considered at highest risk of severe influenza complications, and what medical or preventive interventions are recommended for them?": [
        "All age groups can be affected but there are groups that are more at risk than others.People at greater risk of severe disease or complications when infected are pregnant women, children under 5 years of age, older people, individuals with chronic medical conditions (such as chronic cardiac, pulmonary, renal, metabolic, neurodevelopmental, liver or hematologic diseases) and individuals with immunosuppressive conditions/treatments (such as HIV, receiving chemotherapy or steroids, or malignancy). Health and care workers are at high risk of acquiring influenza virus infection due to increased exposure to the patients, and of further spreading particularly to vulnerable individuals. Vaccination can protect health workers and the people around them.\n\nEpidemics can result in high levels of worker/school absenteeism and productivity losses. Clinics and hospitals can be overwhelmed during peak illness periods.",
        "It's very important that flu antiviral drugs are started as soon as possible to treat patients who are: hospitalized with flu, people who are very sick with flu but who do not need to be hospitalized, and people who are at increased risk of serious flu complications based on their age or underling health conditions, if they develop flu symptoms. For example, people with asthma and chronic lung disease, diabetes, or heart disease are at higher risk, as well as pregnant women.\n\nAlthough patients with mild illness who are not at higher risk for flu complications may also be treated with antiviral drugs, most do not need to be. Children can take flu antiviral drugs, though this varies by medication. Oseltamivir is recommended by CDC for treatment of flu in children beginning from birth and the American Academy of Pediatrics (AAP) recommends oseltamivir for treatment of flu in children 2 weeks old or older.",
        "The first and most important step in preventing flu is to get a flu vaccine each year. Flu vaccine has been shown to reduce flu-related illnesses and the risk of serious flu complications that can result in hospitalization or even death. CDC also recommends everyday preventive actions (like staying away from people who are sick (distancing), covering coughs and sneezes, frequent handwashing, and taking steps for cleaner air) to help slow the spread of germs that cause respiratory (nose, throat, and lungs) illnesses like flu. More information is available about core and additional prevention strategies.",
        "Anyone can get flu (including healthy people), and serious problems related to flu can happen at any age, but some people are at higher risk of developing serious flu-related complications if they get sick. This includes people 65 years and older, people of any age with certain chronic medical conditions (such as asthma, diabetes, or heart disease), people with a body mass index (BMI) of 40 kg/m2 or higher, those who are pregnant, and children younger than five years."
    ],

    "What are the major risk factors and early symptoms of myocardial infarction, and how does public awareness impact treatment outcomes?": [
        "There are many potential arrhythmic complications in the early post MI period. Appropriate treatment is often dependent on patient factors including timing of presentation, left ventricular function, clinical status, and co-morbidities. Some peri-infarct arrhythmias including VAs, atrial fibrillation, and persistent high-grade AV block, require treatment. Other arrhythmias such as sinus bradycardia and sinus tachycardia, as well transient high- grade AV block may require acute treatment, but often resolve with reperfusion and time. In all cases, the ability to recognize these rhythms, understand their likelihood, and appreciate the associated risks improves preparedness and hopefully outcomes for these vulnerable patients.",
        "All patients with suspected acute coronary syndrome should be admitted urgently to hospital because there is a risk of death or recurrent myocardial ischaemia during the early unstable phase. Appropriate medical therapy can reduce the incidence of these complications by at least 60%. The key elements of immediate in-hospital management are shown in Fig. 16.69. Patients should ideally be managed in a dedicated cardiac unit, where the necessary expertise, monitoring and resuscitation facilities are available. Clinical risk factor analysis using tools such as the GRACE score (see Fig. 16.61) should be performed to identify patients that should be selected for intensive therapy, and specically early inpatient coronary angiography (thresholds vary, but a score of 140 points or more supports early intervention). If there are no complications and risk factor analysis shows that angiography is not required, the patient can be mobilised from the second day and discharged after 2–3 days. Low-risk patients without spontaneous angina may be considered for an exercise tolerance test 4-6 weeks after the acute coronary syndrome. This will help to identify those individuals who may require further investigation, and may help to boost the confidence of the remainder.",
        "In a major study in the USA, one-third of 434,877 subjects with confirmed diagnosis of myocardial infarction did not have chest pain on presentation. When chest pain is not the main presenting complaint, patients may be confused about the severity of their symptoms and thus postpone seeking treatment. From a global viewpoint, the level of knowledge of signs and symptoms of heart attack in Singapore is comparable to USA and Canada. From 2005 to 2009, the Centers for Disease Control in USA collated data nationwide via telephone interviews with a total of 103,262,115 respondents on heart attack knowledge. Their aim was to compare knowledge between the nonrural and rural populations. Singapore is a city-state, thus singling out their nonrural population analysis for comparison; their results showed a higher knowledge, with 92.8% identifying at least one symptom correctly, compared to 85.1% in Singapore. They found the more educated and younger adults (19 to 65 years old) to have higher knowledge, congruent with our study. Unlike ours, there was also a racial and gender discrepancy with Hispanics and women scoring lower. In Vancouver, Canada, an urban study published in 2008 showed more similar results to ours, with 83.6% identifying at least one out of 10 symptoms correctly."
    ],

    "How can early diagnosis, evolving clinical definitions, and global prevention strategies improve sepsis outcomes and reduce mortality worldwide?": [
        "Implementing preventive measures against infections, such as good hygiene practices, ensuring access to vaccination programmes, improved sanitation and water quality and availability, and other infection prevention and control best practices both in the community and health care settings, are key steps in reducing the occurrence of sepsis. Early diagnosis and timely and appropriate clinical management of sepsis, such as optimal antimicrobial use and fluid resuscitation, are crucial to increase the likelihood of survival. Even though the onset of sepsis can be acute and poses a short-term mortality burden, it can also",
        "With the improvement of medical treatment and scientific research, the therapy and prognosis of sepsis have become more standardized. Globally, the prevalence and mortality of sepsis have improved significantly. Unfortunately, however, there are still no specific diagnostic markers or therapeutic tools for sepsis. Early studies have successively demonstrated that high lactate levels predict a poor prognosis for patients with sepsis, the idea that sepsis is associated with reduced morbidity and mortality after correction of high lactate during hyperinflammation is well established, and early monitoring and reduction of serum lactate has been included in the latest international guidelines for the management of sepsis and septic shock. These ideas suggest that lactate somehow mediates immune disorders (especially immune paralysis) in sepsis patients, so some investigators have suggested that lactate is an immunosuppressive molecule. The discovery of lactylation modifications apparently explains at the epigenetic level the underlying cause of lactate regulation of immune status in sepsis.",
        "Despite significant advances in understanding pathophysiology and supportive treatment options, mortality from sepsis and septic shock remains very high. It is estimated that one in five patients diagnosed with sepsis dies. Mortality is also high in patients in whom transient improvement is achieved through intensive treatment, and the reason for this is most often complications associated with existing diseases or irreversible impairment of the function of one of the vital organs. Sepsis and septic shock have been identified as important public health issues, prompting intensive care professionals to develop guidelines, the SSC, that could guide clinicians in treating septic patients. The campaign to introduce the guidelines was initiated at a meeting in Barcelona, based on all previous guidelines based on evidence and renewed research on more than 30,000 patients. The main idea was to define global criteria for early detection of sepsis with recommendations for the implementation of certain therapeutic procedures in order to improve their effectiveness and reduce mortality by 25% over five years. The guidelines have undergone many changes over the years as part of the latest clinical research and new pathophysiological findings on sepsis. The first original guidelines were published in 2004, and have been updated and supplemented on several occasions to date in 2008, 2012, 2016 and 2018. The last renewal and amendment of the SSC was performed in 2021. Evidence based methodology was used in the renewal of the guidelines.",
        "Sepsis is a life-threatening organ dysfunction caused by an unregulated response of a host. Septic shock is its most severe form. It is manifested by a drop in blood pressure, which decreases tissue perfusion pressure, causing hypoxia that is characteristic of shock. Sepsis is still one of the leading causes of mortality worldwide. Its incidence has increased since the first consensus definitions were established in 1991. Raising sepsis awareness, its significance and the need for better treatment, has led to an improvement in in defining sepsis and the development of guidelines for its treatment. The first guidelines were published in 2004, the second 2008, the third 2013, the fourth 2016, and the last revised guidelines appeared in 2021. This paper will describe the previous and new definitions of sepsis and septic shock, the previous guidelines for the recognition and treatment, and the latest recommendations for treatment. Timely diagnosis is crucial for the outcomes for patients with sepsis and septic shock. The fact is that the sepsis care bundles have been modified to increasingly shorter time determinants, which emphasizes the importance of emergency physicians, who frequently first recognize and begin emergency treatment of septic patients.",
        "Sepsis is a significant cause of maternal, neonatal and child mortality. Consequently, combating sepsis will contribute to achievement of Sustainable Development Goals (SDGs) targets 3.8 on quality of care, and 3.1 and 3.2 by improving mortality rates in these vulnerable populations. Sepsis can also ultimately lead to death in patients affected by HIV, tuberculosis, malaria, and other infectious diseases that are included in target 3.3. The prevention and/or appropriate diagnosis and management of sepsis is also linked to adequate vaccine coverage, quality universal health coverage, capacity to comply with the International Health Regulations, preparedness, and water and sanitation services. The challenge, however, remains how to achieve universal prevention, diagnosis and management of sepsis."
    ],

    "Which emerging therapies, including gene therapy and molecular targeted approaches, show promise in revolutionizing overall cancer treatment paradigms?": [
        "Despite the availability of technological advances in traditional anti-cancer therapies, there is a need for more precise and targeted cancer treatment strategies. The wide-ranging shortfalls of conventional anticancer therapies such as systematic toxicity, compromised life quality, and limited to severe side effects are major areas of concern of conventional cancer treatment approaches. Owing to the expansion of knowledge and technological advancements in the field of cancer biology, more innovative and safe anti-cancerous approaches such as immune therapy, gene therapy and targeted therapy are rapidly evolving with the aim to address the limitations of conventional therapies. The concept of immunotherapy began with the capability of coley toxins to stimulate toll-like receptors of immune cells to provoke an immune response against cancers. With an in-depth understanding of the molecular mechanisms of carcinogenesis and their relationship to disease prognosis, molecular targeted therapy approaches, that inhibit or stimulate specific cancer-promoting or cancer-inhibitory molecules respectively, have offered promising outcomes. In this review, we evaluate the achievement and challenges of these technically advanced therapies with the aim of presenting the overall progress and perspective of each approach. Cells acquire a cancerous phenotype due to a multitude of aberrant changes that manifest at the levels of proteins, RNA, or DNA. The year 2020 witnessed staggering statistics from the World Health Organization (WHO)-one-sixth of global deaths were attributed to cancer, underscoring the urgent necessity for safer, personalized, and more effective treatment modalities. While conventional anticancer methods such as surgery, radiotherapy, and hormonal therapy have shown advancements, the realm of cancer therapeutics is abuzz with exploration aimed at enhancing survival rates. Within this landscape, emerging treatment avenues including immunotherapy, gene therapy, and molecular targeted therapy are offering promising prospects. These innovative therapeutic paradigms have historical roots, but it’s the availability of comprehensive genomic and individualized data that has truly refined their applications.",
        "Until now multiple therapeutic strategies have been devised to treat cancer. While conventional therapies have served as a cornerstone in cancer management, their limitations, such as the development of treatment resistance and tumor relapse, have paved the way for advanced therapeutic approaches. These innovative strategies aim to address the challenges posed by traditional treatments. Recognizing the pivotal role of the immune system in cancer progression, scientists have explored the modulation of immune cells to achieve targeted immune responses against circulating cancer cells. The inherent heterogeneity of cancer cells and the concept of personalized medicine have further propelled the exploration of gene and targeted molecular therapies in cancer treatment. The integration of advanced therapies has not only enhanced the ability of clinicians to manage cancer effectively but has also provided researchers with the opportunity to refine these approaches for optimal anti-cancer solutions. Despite inherent shortcomings in each proposed solution, the current management of cancer involves the judicious utilization of both traditional and advanced approaches to treat various forms of cancer. The inclusion of advanced therapies has shown significant improvements in cancer management, leading to enhanced survival rates. However, the ultimate goal of achieving disease-free survival for all cancer patients, irrespective of tumor grade and stage, remains a formidable challenge yet to be fully realized.",
        "These innovative therapeutic paradigms have historical roots, but it’s the availability of comprehensive genomic and individualized data that has truly refined their applications. The core objective of these groundbreaking treatments is to overcome the limitations inherent in traditional anticancer approaches—adverse treatment effects and long-term side effects. In spite of these strides, cancer stands as the second leading cause of mortality, prompting an urgent quest for precise, targeted anticancer interventions to improve tolerance and mitigate both immediate and enduring side effects. The pursuit of better outcomes steers oncologists towards a strategy of integrated disease management, entailing dynamic treatment regimens that optimize cancer management. This article delves into anti-cancer therapeutic methods—immunotherapy, gene therapy, and molecular targeted therapy—tracing their historical development, assessing present progress, and outlining the potential they hold for the future.",
        "Looking at this, we have various forms of “targeted” therapy directed at specific single molecular targets or a class of molecular targets in cancer cells. With targeted therapy, the specific mechanism of action of the drug results in an increase in its therapeutic index. Currently, the two major classes of targeted therapy are the small molecule tyrosine kinase inhibitors and monoclonal antibodies (MABs). Imatinib, for instance, has been amazingly successful as a targeted agent directed against several members of a class of enzyme known as tyrosine kinases, and by that mechanism, it has been phenomenally successful as a treatment of chronic myeloid leukemia and gastrointestinal stromal tumors. Tamoxifen is a targeted therapy directed at the estrogen receptor (ER), and it still remains a mainstay of treatment for ER?positive breast cancers, along with a newer class of drugs known as aromatase inhibitors. Cancer researchers work on developing new and more effective surgery techniques, radiotherapy and chemotherapy drugs all the time. Biological therapies such as MABs, cancer vaccines, and gene therapies are all active areas of research. There are various anti?angiogenic drugs that can stop cancers from growing the blood vessels that they need. Research is also looking into developing cost?effective ways of screening for the different common cancers so they can be diagnosed early enough to be cured. Avoiding immune destruction is now considered a hallmark of cancer, and the immunotherapy arena has exploded with the recent advances demonstrating an improvement in survival and a durability of response in patients with different cancer types, like in melanoma, which translates into an improved overall survival benefit. To name those immunotherapeutic strategies that include the adoptive transfer of ex vivo activated T cells, immunomodulatory MABs, and cancer vaccines. Advances in molecular pathology will provide the means to identify the targets and will be used to subtype tumors and will provide predict response to therapy and provide prognostic information.",
        "The combination of next-generation sequencing and advanced computational data analysis approaches has revolutionized our understanding of the genomic underpinnings of cancer development and progression. The coincident development of targeted small molecule and antibody-based therapies that target a cancer’s genomic dependencies has fuelled the transition of genomic assays into clinical use in patients with cancer. Beyond the identification of individual targetable alterations, genomic methods can gauge mutational load, which might predict a therapeutic response to immune-checkpoint inhibitors or identify cancer-specific proteins that inform the design of personalized anticancer vaccines. Emerging clinical applications of cancer genomics include monitoring treatment responses and characterizing mechanisms of resistance. The increasing relevance of genomics to clinical cancer care also highlights several considerable challenges, including the need to promote equal access to genomic testing."
    ],


    "Which innovations in smart healthcare systems are improving disease diagnosis and clinical decision-making?": [
    "In the long term, AI systems will become more intelligent, enabling AI healthcare systems achieve a state of precision medicine through AI-augmented healthcare and connected care. Healthcare will shift from the traditional one-size-fits-all form of medicine to a preventative, personalised, data-driven disease management model that achieves improved patient outcomes (improved patient and clinical experiences of care) in a more cost-effective delivery system.Connected/augmented care AI could significantly reduce inefficiency in healthcare, improve patient flow and experience, and enhance caregiver experience and patient safety through the care pathway; for example, AI could be applied to the remote monitoring of patients (eg intelligent telehealth through wearables/sensors) to identify and provide timely care of patients at risk of deterioration. In the long term, we expect that healthcare clinics, hospitals, social care services, patients and caregivers to be all connected to a single, interoperable digital infrastructure using passive sensors in combination with ambient intelligence.",
    "* Disease prediction: Machine learning can be used to find trends, create connections, and make conclusions based on large data sets. Data engineers can use information to prevent disease outbreaks in communities and track habits that lead to disease. * Biomedical data visualizations: You can use machine learning to create 3-D visualizations of biomedical data such as RNA sequences, protein structures, and genomic profiles. * Improve diagnoses: Identify previously unrecognizable symptom patterns and compare them with larger data sets to diagnose diseases earlier in their development. * Accurate health records: Keep patient records updated, accurate, and easy to transfer between clinics, physicians, and medical staff by employing machine learning to filter out errors and blanks. * AI-assisted surgery: Support surgeons by performing complex tasks during surgery, giving surgeons a better view of their work area, and modeling how to complete procedures. * Personalized treatment options: You can use machine learning to analyze multi-modal data and make patient-tailored decisions based on possible treatment options. * Medical research and clinical trial improvement: You can use machine learning to enhance the selection of participants for clinical trials, data collection procedures, and analysis of the results. * Develop medications: You can use machine learning to identify potential pathways for new medicines and develop innovative drugs to treat varying medical conditions.",
    "The applications of various disease diagnosis in smart healthcare are related to automated decision making. If the performance of the classification algorithm is good in terms of overall accuracy and testing time, it may ultimately replace the role of medical doctors in disease diagnosis (and medical doctors can devote their time majorly in complicated surgery). The second case is that the classification algorithm will be utilized as rapid test (fair overall accuracy and rapid decision) for low-cost and large-scale screening."
    ],

    "What key social and environmental factors have been identified as influencing mental health outcomes, and how do recent studies suggest addressing them?": [
        "Social and Environmental Factors reflected the societal factors external to the individual that affect mental health. Although many respondents listed the basic necessities for general health/mental health (eg, housing, food security, access to health services, equitable access to public resources, childcare, education, transportation, support for families, respect for diversity, opportunities for building resilience, selfesteem, personal and social efficacy, growth, meaning and purpose, and sense of safety and belonging, and employment), some also recommended approaches to achieving social equity (eg, “mental health needs to be protected by applying antiracism, antioppression, antidiscrimination lens to prevention and treatment”) (figure 1, Direction D). A distinct category of human rights developed from responses to the third open-ended question (eg, “What is missing?”)",
        "In contrast, Social and Environmental Factors reflected respondents’ emphasis on factors that are external to the individual and which can influence the core concepts of mental health. Many respondents reiterated the basic necessities for general health/mental health, similar to the foundations of Maslow’s hierarchy of needs, and their recommendations for achieving social equity. Descriptions of the core concepts of mental health were highly influenced by respondents’ Positionality and Paradigms/Theories/Models of reference, which often propelled the discourse of “What is mental health?” in opposing directions. The debate as to whether mental health and illness are distinct constructs, or points of reference on a continuum of being, was a common theme. Respondents were either, adamant in asserting the distinction between the descriptive or prescriptive nature of the core concepts, or, ardent in integrating them, producing ideas such as describing mental health as a life free of poverty, discrimination, oppression, human rights violations and war. Respondents’ made repeated references to human rights, suggesting that a basic standard, analogous to a legal definition, is required, and that ‘a human rights, political, economic and ecosystem perspective’ should be included. Again, in the tradition of Hume’s ‘ought–is’ distinction, several respondents cautioned that problems of living, such as ‘poverty, vices and social injustices’ should not be defined ‘as medical problems’. The significance of this issue cannot be understated: while we asked respondents what the core concepts of mental health are, overwhelmingly they answered in terms of what they should be. This finding is similar to other issues in public health policy that address instances of ‘conflating scientific evidence with moral argument’.",
        "With respect to the factors related to mental health, the following aspects can be considered: biological (age, sex, genetics, and special conditions), psychological (personality traits, values, motivations, and aspects of self-regulation), social (educational level, gender, socioeconomic status, marital status, occupation, and family composition), and environmental (stressful, challenging, hostile, among others); by the contributions of several authors [for example, DeNeve and Cooper, 1998; Lyubomirsky et al., 2005; Keyes et al., 2010; American Psychiatric Association, 2013; Lim, 2017; World Health Organization (WHO), 2018; among others]. In relation to mental health care options, World Health Organization (WHO) (2009) made a distinction between informal services (self-care and community care), and formal services (primary care, community services, psychiatric in general hospitals, long-stay facilities, and specialized services). However, it emphasized that self-care is essential and occurs simultaneously with other services, as the person self-manages, without the intervention of a professional, their mental health problems; in addition to promoting and fostering their recovery, and better mental health. Based on the above, it is clear that mental health is a complex concept, which, from different approaches, has given rise to a proliferation of conceptual notions, making it difficult to measure and create strategies for improvement. In this sense, it is necessary to reduce the gap by analyzing the input toward a proposal for a unifying concept. Therefore, the aim of the present study was to analyze different conceptual contributions regarding mental health through the Systematic Literature Review (SLR). Literature reviews related to the definition of mental health are few, mostly focused on analyzing specific related aspects, for example, mental health in specific populations (Suhaiban et al., 2019), the type of care people receive (Bakker et al., 2016), related factors (Shalaby and Agyapong, 2020), among others. The concept of mental health was identified by the work of Muñoz et al. (2016), which was based on an analysis of positive mental health, and the work done by Wang and Lai (2022), in which they used two paradigms as references in their analysis: the absence of disease and positive mental health. As can be seen, the literature reviews on mental health are diverse and polysemic.",
        "These are aspects related to the person that has an association with one’s mental health status. With respect to these aspects, biological factors (age, sex, genetics, and special conditions), psychological factors (personality traits, values, motivations, and self-regulation), social factors (educational level, gender, socioeconomic status, marital status, occupation, and family composition), and environmental factors (stressors, discrimination, bullying, challenging, hostile, among others) can be considered in accordance with the contributions of various authors [e.g., DeNeve and Cooper, 1998; Lyubomirsky et al., 2005; Keyes et al., 2010; American Psychiatric Association, 2013; Lim, 2017; World Health Organization (WHO), 2018; among others]. Based on this information, Figure 3 shows the explanatory model of the integral definition of mental health as a result of the Grounded Theory approach (Strauss and Corbin, 1990). The present definition and its respective explanatory model bring value to the field of mental health because it is a concept that integrates contributions from various disciplines such as psychology, sociology, psychiatry, philosophy and education; unites diverse paradigms such as positive mental health, mental health based on the absence of illness and mental health based on a state of balance; focuses on self-care, recognized by World Health Organization (WHO) (2009) as an essential element of care for mental health; seeks to empower the person to manage one’s mental health problems; is person-centered by considering the person’s values and motivations; is based on intrapersonal and interpersonal aspects, two distinct dimensions that converge to give meaning to mental health; is relative to the person by taking into account various factors that vary according to development, context and personal characteristics; and prevents stigmatization by focusing the concept on an internal process of personal balance, rather than a state of complete wellbeing and absence of mental disorder that could be utopian. The present research is an approach that seeks to understand the concept of mental health from different disciplines; however, it is recognized that the keywords could exclude other disciplines that contribute to its understanding; therefore, it is recommended to analyze the concept from other disciplines not considered in the research. In addition, the complexity of the concept and its associated factors is recognized; therefore, it is suggested that future research must go deep into the study of the existing relationships between the elements that conform to the proposed explanatory model of mental health in different contexts. The original contributions presented in the study are included in the article/Supplementary material, further inquiries can be directed to the corresponding author. MC-S conducted a systematic review of mental health paradigms. JR-M provided a context in mental health’s different paradigms. The authors worked together to edit and integrate contents in the manuscript. All authors contributed to the article and approved the submitted version."
    ],

    "What role do digital technologies and mobile platforms play in improving diabetes management and patient engagement?": [
        "These interventions leverage innovative technological solutions to enhance outreach, education, monitoring, and management of individuals at risk for T2DM. By incorporating mobile applications, wearable devices, and online platforms, digital health technologies provide personalized and real-time interventions, promoting lifestyle modifications and behavior change. These tools facilitate continuous glucose monitoring, physical activity tracking, and dietary management, empowering individuals to take an active role in their health. Moreover, digital interventions enable healthcare providers to remotely monitor patients, offer timely if not real-time feedback, and tailor interventions based on individual progress. The integration of data analytics and machine learning further refines predictive models, identifying high-risk populations and customizing intervention strategies. In essence, digital health technology interventions not only complement but also transform conventional approaches to T2DM prevention, fostering a more dynamic, individualized, and proactive healthcare paradigm. Implications for practice. This study provided evidence for the generalized efficacy of health technologies in preventing diabetes by reducing risk related outcomes as 5 studies within this review demonstrated statistically significant reductions in HbA1c and diabetes incidence. Further research is needed to translate this data into clinical settings. The feasibility of digitally mediated DPPs is supported by the gradual modernization of the healthcare system and public accessibility to technology but this can be better understood with more pragmatic trials involving diverse populations or interventions that are culturally adapted (which showed a positive outcome in one study.29 We also provided evidence of appropriateness, as diabetes prevention involves self-management and literature which identifies mobile devices efficacy in increasing patient engagement39,40 and the high retention rates of studies within this review support this. Evidence of meaningfulness was unavailable as patient-rated outcomes were not examined in this review and effectiveness is difficult to determine with the moderate to low quality evidence of the chosen studies.",
        "The results showed that more than half of the studies showed significant outcomes. For instance, some interventions led to decreased weight, lower glucose concentrations, or reduced incidence of T2DM. Computer-based interventions and mobile-based interventions (including text messages, mobile apps, and telehealth) were particularly effective in improving these outcomes. In conclusion, our review suggests that digital health technologies can be effective in preventing diabetes and improving related outcomes. However, we note that more research is needed, especially looking at diverse populations and longer study durations, to confirm the clinical feasibility of these digital interventions for diabetes prevention. This is a promising step forward in using technology to tackle the growing diabetes epidemic, offering new ways to support individuals at risk and improve their health Outcomes.",
        "Key objectives in these programmes include achieving a minimum weight loss of 7 % and engaging individuals in moderate-intensity physical activity for a minimum of 150 min every week. While meta-analyses of studies have indicated that lifestyle interventions are effective in preventing or delaying the progression to T2DM, implementing these interventions on a large scale remains challenging. Some key challenges include the difficulty in monitoring the activity and dietary practice of individuals with prediabetes, maintaining high levels of motivation and adherence to the prescribed interventions, as well as the delivery of relevant and timely information",
        "One solution to overcome these challenges is through the use of digital technologies - electronic systems designed to provide service remotely which have become increasingly prevalent in healthcare. These technologies include telehealth, mobile health (mHealth), game-based support, social platforms, patient portals, as well as wearable devices that collect health data and deliver information. Digital technologies are now used in the monitoring and management of non-communicable diseases including hypertension, T2DM, weight and diet management, as well as to improve medication adherence. The incorporation of digital technologies as remote components has been promising as scalable tools to improve health outcomes as they enhance efficacy, efficiency, accessibility, and allows for personalisation, potentially complementing lifestyle interventions which are often a key challenge in implementation. However, while there is considerable enthusiasm for the innovative role of digital health interventions, the World Health Organization highlighted that it is equally important to generate high-quality evidence and evaluate intervention-contributing effects to ensure that resources are not diverted to ineffective approaches. The lack of comprehensive pooled data from randomised-controlled trials (RCTs) precludes the implementation of digital technologies as an integral digital approach for preventing or delaying diabetes among high-risk groups at a larger scale. While several reviews on the use of digital technologies have been published, these review only focus on people with type 2, type 1 and gestational diabetes."
    ],

    "What is insulin, how does it regulate blood glucose, and how has insulin therapy evolved over time?": [
        "Insulin has been available for the treatment of diabetes for almost a century, and the variety of insulin choices today represents many years of discovery and innovation. Insulin has gone from poorly defined extracts of animal pancreata to pure and precisely controlled formulations that can be prescribed and administered with high accuracy and predictability of action. Modifications of the insulin formulation and of the insulin molecule itself have made it possible to approximate the natural endogenous insulin response. Insulin and insulin formulations had to be designed to produce either a constant low basal level of insulin or the spikes of insulin released in response to meals. We discuss how the biochemical properties of endogenous insulin were exploited to either shorten or extend the time-action profiles of injectable insulins by varying the pharmacokinetics (time for appearance of insulin in the blood after injection) and pharmacodynamics (time-dependent changes in blood sugar after injection). This has resulted in rapid-acting, short-acting, intermediate-acting, and long-acting insulins, as well as mixtures and concentrated formulations. An understanding of how various insulins and formulations were designed to solve the challenges of insulin replacement will assist clinicians in meeting the needs of their individual patients.",
        "Insulin is an anabolic hormone (i.e. it promotes the storage of nutrients) and has pleiotropic effects on glucose, fat and protein metabolism (Box 21.1). It also stimulates cell proliferation and reduces apoptosis. The actions of insulin are mediated through the cell membrane insulin receptor. The receptor belongs to the tyrosine kinase superfamily and is coupled to an extremely complicated intracellular signalling network. A key action of insulin is to lower blood glucose and, in part, this is mediated by promoting the uptake of glucose into muscle and adipose tissue; at a cellular level, the binding of insulin to its receptor causes cell membrane glucose transporter GLUT4 to translocate from the cytoplasm to the cell membrane, permitting the uptake of glucose into adipose and muscle cells.",
        "This last century has been a time of change and innovation in the field of insulin therapy, starting with the isolation of insulin, the purification and concentration of animal pancreatic extracts, the development of formulations with protracted duration of action, and the progression to human insulin and modified insulin analogs made with recombinant DNA technology. The landscape of insulins available today also includes insulin mixtures, concentrated insulins, and insulins with alternate routes of administration, providing a wide array of options for people living with diabetes",
        "Replacement insulin therapy should mimic the body’s own insulin response as closely as possible. Great strides have been made in achieving this goal through innovation and the use of biotechnology, including recombinant DNA technology, protein engineering, formulation strategies, and advances in manufacturing. Of course, true insulin replacement requires the feedback control on insulin levels, such as is provided naturally by healthy beta cells in response to changes in blood glucose. For this, we have evolved from using episodic self-monitored blood glucose values to CGM technology, which could enable more accurate, feedback-based insulin replacement with pens and hybrid closed-loop pump systems. Advances in insulin’s molecular properties through new analogs, coupled with advances in glucose monitoring and dosing algorithms, will continue to make insulin therapy safer and more effective for people with diabetes.",
        "Newer, faster-acting insulins, called ultrarapid acting, are now entering the market. These insulins have an onset of action faster than rapid-acting analog insulins, allowing dosing to occur at the start of or even during a meal to better control postprandial glucose peaks. The first ultrarapid insulin, marketed as Fiasp®, was approved by the FDA in 2017 (30). Fiasp® contains insulin aspart formulated with 2 additional excipients, L-arginine and niacinamide. L-arginine acts as a stabilizing agent, while niacinamide accelerates absorption at the site of injection (31). A comparison of the PK and PD profiles of Fiasp® and insulin aspart showed an approximately 5- to 6-minutes faster onset of action with Fiasp® (Fig. 6) (32). Fiasp® is recommended to be injected at the start of a meal or within 20 minutes after starting a meal (30)."
    ]
}

with open(MANUAL_BASELINE_PATH, "w") as f:
    json.dump(manual_baseline, f, indent=2)

## Evaluation

In [13]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import ndcg_score
from rouge_score import rouge_scorer
from typing import Dict, List, Any
import itertools
import numpy as np

# Sentence-level encoder for semantic similarity
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ROUGE-L scorer for lexical overlap
rouge_scorer_fn = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Weights for hybrid similarity: embedding cosine + ROUGE-L F1
EMB_WEIGHT = 0.9
ROUGE_WEIGHT = 0.1
SIM_THRESHOLD = 0.6  # you can tune this if needed


def hybrid_similarity_matrix(retrieved_texts: List[str], golden_texts: List[str]) -> np.ndarray:
    if not retrieved_texts or not golden_texts:
        return np.zeros((len(retrieved_texts), len(golden_texts)), dtype=float)

    # Embedding-based cosine similarity
    emb_ret = model.encode(retrieved_texts, convert_to_tensor=True)
    emb_gold = model.encode(golden_texts, convert_to_tensor=True)
    cos_sim_matrix = util.cos_sim(emb_ret, emb_gold).cpu().numpy()  # (retrieved, golden)

    # ROUGE-L F1 similarity
    rouge_matrix = np.zeros_like(cos_sim_matrix, dtype=float)
    for i, r_txt in enumerate(retrieved_texts):
        for j, g_txt in enumerate(golden_texts):
            scores = rouge_scorer_fn.score(g_txt, r_txt)  # target, prediction
            rouge_matrix[i, j] = scores["rougeL"].fmeasure  # already in [0,1]

    # Hybrid similarity
    sim_matrix = EMB_WEIGHT * cos_sim_matrix + ROUGE_WEIGHT * rouge_matrix
    return sim_matrix


def compute_metrics(
    retrieved_texts: List[str],
    golden_texts: List[str],
    threshold: float = SIM_THRESHOLD,
) -> Dict[str, float]:
    if not retrieved_texts or not golden_texts:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "coverage": 0.0,
            "mean_max_similarity": 0.0,
        }

    # Hybrid similarity matrix (cosine + ROUGE-L)
    sim_matrix = hybrid_similarity_matrix(retrieved_texts, golden_texts)  # shape: (retrieved, golden)

    # Max similarity per retrieved and per golden
    max_sim_retrieved = sim_matrix.max(axis=1)
    max_sim_golden = sim_matrix.max(axis=0)

    # Precision/recall/F1 (using hybrid similarity)
    precision = (max_sim_retrieved > threshold).mean()
    recall = (max_sim_golden > threshold).mean()
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    # Coverage and mean-max similarity
    coverage = max_sim_golden.mean()
    mean_max_similarity = max_sim_retrieved.mean()

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "coverage": float(coverage),
        "mean_max_similarity": float(mean_max_similarity),
    }


def evaluate_retrieval_chunk_level(
    chunk_results: Dict[str, Any],
    manual_baseline: Dict[str, List[str]],
    chunks_catalog: List[Dict],
    k: int = 10,
    threshold: float = SIM_THRESHOLD,
) -> Dict[str, Any]:

    # Build chunk_id -> text lookup
    chunk_lookup = {c["chunk_id"]: c["text"] for c in chunks_catalog}

    # Initialize accumulators
    precisions, recalls, f1s = [], [], []
    coverages, mean_sim = [], []
    mrr_scores, ndcg_scores = [], []

    qualitative: Dict[str, Any] = {}

    for query, golden_chunks in manual_baseline.items():
        ret = chunk_results.get(query, {"chunk_ids": [], "doc_ids": []})
        retrieved_chunks = ret.get("chunk_ids", [])[:k]
        retrieved_texts = [chunk_lookup[c] for c in retrieved_chunks if c in chunk_lookup]
        gold_texts = list(golden_chunks)

        # Compute metrics (hybrid similarity)
        sem = compute_metrics(retrieved_texts, gold_texts, threshold=threshold)
        precisions.append(sem["precision"])
        recalls.append(sem["recall"])
        f1s.append(sem["f1"])
        coverages.append(sem["coverage"])
        mean_sim.append(sem["mean_max_similarity"])

        # If nothing to compare, MRR and nDCG are 0 for this query
        if not retrieved_texts or not gold_texts:
            mrr_scores.append(0.0)
            ndcg_scores.append(0.0)
            qualitative[query] = {
                "retrieved_chunks": retrieved_chunks,
                "retrieved_texts": retrieved_texts,
                "golden_chunks": gold_texts,
                "metrics": sem,
                "mrr": 0.0,
                "ndcg": 0.0,
            }
            continue

        # Hybrid similarity for this query
        sim_matrix = hybrid_similarity_matrix(retrieved_texts, gold_texts)
        max_sim_per_retrieved = sim_matrix.max(axis=1)

        # Traditional MRR, but using hybrid similarity + threshold
        rank = None
        for i, sim_val in enumerate(max_sim_per_retrieved):
            if sim_val > threshold:
                rank = i + 1  # 1-based
                break
        mrr_scores.append(1.0 / rank if rank else 0.0)

        # nDCG using hybrid similarity scores
        y_score = max_sim_per_retrieved.tolist()
        y_true = [1 if s > threshold else 0 for s in y_score]
        ndcg_scores.append(ndcg_score([y_true], [y_score]))

        qualitative[query] = {
            "retrieved_chunks": retrieved_chunks,
            "retrieved_texts": retrieved_texts,
            "golden_chunks": gold_texts,
            "metrics": sem,
            "mrr": mrr_scores[-1],
            "ndcg": ndcg_scores[-1],
        }

    # Aggregate results
    final = {
        "Precision": float(np.mean(precisions)) if precisions else 0.0,
        "Recall": float(np.mean(recalls)) if recalls else 0.0,
        "F1": float(np.mean(f1s)) if f1s else 0.0,
        "Coverage": float(np.mean(coverages)) if coverages else 0.0,
        "MeanMaxSimilarity": float(np.mean(mean_sim)) if mean_sim else 0.0,
        "MRR": float(np.mean(mrr_scores)) if mrr_scores else 0.0,
        "nDCG": float(np.mean(ndcg_scores)) if ndcg_scores else 0.0,
        "examples": qualitative,
    }

    return final

In [23]:
import time

# Example: measure end-to-end latency
start_time = time.time()

evaluation = evaluate_retrieval_chunk_level(
    chunk_results=results_chunks,
    manual_baseline=manual_baseline,
    chunks_catalog=chunks,
    k=TOP_K
)

with open(EVAL_PATH, "w") as f:
    json.dump(evaluation, f, indent=2)

print(json.dumps(evaluation, indent=2))

{
  "Precision": 0.7613636363636364,
  "Recall": 0.790909090909091,
  "F1": 0.7664335114368073,
  "Coverage": 0.7151749952650664,
  "MeanMaxSimilarity": 0.6822900686989857,
  "MRR": 0.8181818181818182,
  "nDCG": 0.9090909090909091,
  "examples": {
    "Based on the collective perspective of recent literature, what are the major biological, technological, and sociopolitical barriers that still hinder full malaria eradication despite modern advances?": {
      "retrieved_chunks": [
        "Malaria Pathogenesis Danny A.txt__c0004",
        "An Overview of Malaria Transmission Mechanisms, Control,.txt__c0005",
        "An update on Malaria.txt__c0051",
        "An Overview of Malaria Transmission Mechanisms, Control,.txt__c0051",
        "WHO-guidelines-for-malaria.pdf__c2108",
        "WHO-guidelines-for-malaria.pdf__c1605",
        "WHO-guidelines-for-malaria.pdf__c0171",
        "WHO-guidelines-for-malaria.pdf__c1606"
      ],
      "retrieved_texts": [
        "and vary in total popul

## LLM as Evaluator

In [24]:
from openai import OpenAI
import os
import json
import re
from typing import List, Dict, Any

# ---------- LLM CLIENT SETUP (GROQ) ----------

client = OpenAI(
    api_key="",
    base_url="https://api.groq.com/openai/v1"
)


def extract_json(text: str) -> Dict[str, Any]:

    # 1) Try direct JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) Strip code fences if present
    cleaned = text.strip()

    # Case: ```json ... ```
    fence_match = re.search(r"```json(.*)```", cleaned, re.DOTALL | re.IGNORECASE)
    if fence_match:
        inner = fence_match.group(1).strip()
        try:
            return json.loads(inner)
        except Exception:
            # fall through to other heuristics
            cleaned = inner

    # Case: generic ``` ... ```
    fence_match2 = re.search(r"```(.*)```", cleaned, re.DOTALL)
    if fence_match2:
        inner2 = fence_match2.group(1).strip()
        try:
            return json.loads(inner2)
        except Exception:
            cleaned = inner2

    # 3) Fallback: find the first {...} block and try to parse that
    m = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if m:
        candidate = m.group(0)
        try:
            return json.loads(candidate)
        except Exception as e:
            raise RuntimeError(f"Found JSON-like block but failed to parse: {e}") from e

    # 4) Give up
    raise RuntimeError("No valid JSON object found in model output")


def build_eval_prompt(
    question: str,
    chunk_objs: List[Dict],
    max_chunk_chars: int = 5000
) -> str:

    system = (
        "You are an expert evaluator for a medical retrieval-augmented QA system.\n"
        "You are given:\n"
        "  1) A user question.\n"
        "  2) A set of retrieved evidence chunks from a corpus.\n\n"
        "Your job is to decide:\n"
        "  - Do these chunks contain enough information to fully answer the question?\n"
        "  - How complete is the evidence?\n"
        "  - How well grounded and non-hallucinatory an answer based ONLY on these chunks would be?\n\n"
        "IMPORTANT RULES:\n"
        "  - Use ONLY the content of the chunks as evidence.\n"
        "  - Do NOT rely on outside knowledge.\n"
        "  - If key information is missing, you MUST mark the answer as NOT fully answered.\n\n"
        "Field semantics:\n"
        "  - is_fully_answered: true only if a careful, correct answer can be fully derived from the chunks.\n"
        "  - completeness_score: integer from 1 (very incomplete) to 5 (fully complete).\n"
        "  - grounding_score: integer from 1 (likely hallucinations) to 5 (strongly grounded in chunks).\n"
        "  - relevance_score: integer from 1 (mostly off-topic chunks) to 5 (highly relevant chunks).\n"
        "  - missing_key_points: list major important points needed to answer the question that are NOT supported by the chunks.\n"
        "  - high_level_rationale: 2-4 sentences explaining the scores, referencing chunk_ids when possible.\n"
    )

    header = [
        system,
        "\nUser question:",
        question,
        "\nRetrieved chunks:"
    ]

    lines = header.copy()
    for i, c in enumerate(chunk_objs, start=1):
        txt = c.get("text", "")
        if len(txt) > max_chunk_chars:
            txt = txt[:max_chunk_chars].rsplit(" ", 1)[0] + "..."
        lines.append(
            f"\n[{i}] chunk_id={c.get('chunk_id')} doc_id={c.get('doc_id')}\n{txt}\n"
        )

    schema = (
        "\nNow, ANALYZE the chunks and return ONLY a single JSON format.\n"
        "The JSON MUST be valid according to RFC 8259:\n"
        "  - Use double quotes for all strings.\n"
        "  - Use true/false for booleans.\n"
        "  - Use integers for the score fields.\n"
        "  - NO comments, NO trailing commas, NO code fences, NO extra text.\n\n"
        "Return exactly one JSON object with the following keys:\n\n"
        "Example (structure only, you MUST fill in real values):\n"
        "{\n"
        "  \"question\": \"<your restated question>\",\n"
        "  \"is_fully_answered\": true,\n"
        "  \"completeness_score\": 4,\n"
        "  \"grounding_score\": 5,\n"
        "  \"relevance_score\": 5,\n"
        "  \"missing_key_points\": [\n"
        "    \"<point 1>\",\n"
        "    \"<point 2>\"\n"
        "  ],\n"
        "  \"high_level_rationale\": \"<2-4 sentences explanation>\"\n"
        "}\n\n"
        "Output constraints:\n"
        "- Do NOT wrap the JSON in ``` or any other formatting.\n"
        "- Do NOT add any explanation before or after the JSON.\n"
        "- The first character of your reply must be '{' and the last character must be '}'.\n"
    )
    lines.append(schema)
    return "\n".join(lines)

def call_llm(prompt: str) -> str:
    """
    Call the LLM and return raw text output.
    """
    response = client.responses.create(
        model="llama-3.3-70b-versatile",
        input=prompt,
        max_output_tokens=700,
        temperature=0.0
    )
    return response.output_text

chunk_lookup: Dict[str, Dict] = {c["chunk_id"]: c for c in chunks}

llm_eval_results: Dict[str, Any] = {}

for idx, (question, res) in enumerate(results_chunks.items(), start=1):
    safe_print(f"\n=== [{idx}/{len(results_chunks)}] Evaluating question ===")
    cid_list = res.get("chunk_ids", [])[:TOP_K]
    retrieved_chunk_objs = [chunk_lookup[cid] for cid in cid_list if cid in chunk_lookup]

    if not retrieved_chunk_objs:
        safe_print("  No retrieved chunks found; marking as empty.")
        llm_eval_results[question] = {
            "question": question,
            "is_fully_answered": False,
            "completeness_score": 1,
            "grounding_score": 1,
            "relevance_score": 1,
            "missing_key_points": ["No chunks were retrieved."],
            "high_level_rationale": "No evidence chunks available for evaluation."
        }
        continue

    prompt = build_eval_prompt(question, retrieved_chunk_objs, max_chunk_chars=5000)

    try:
        raw_text = call_llm(prompt)
        
        safe_print("  Raw LLM output (truncated):")
        safe_print("  " + raw_text[:400].replace("\n", " ") + ("..." if len(raw_text) > 400 else ""))

        parsed = extract_json(raw_text)
        llm_eval_results[question] = parsed

        safe_print("  Parsed JSON:")
        safe_print(json.dumps(parsed, indent=2))

        safe_print("  ✓ LLM evaluation completed")

    except Exception as e:
        safe_print("  ✗ LLM evaluation failed:", e)
        entry = {"error": str(e)}
        if "raw_text" in locals():
            entry["raw_output"] = raw_text
        llm_eval_results[question] = entry

end_time = time.time()

latency_seconds = end_time - start_time
# print(f"End-to-end latency: {latency_seconds:.3f} seconds")

# Save evaluator outputs
LLM_EVAL_PATH = "rag_llm_eval_completeness.json"
with open(LLM_EVAL_PATH, "w") as f:
    json.dump(llm_eval_results, f, indent=2)

safe_print(f"\nSaved LLM evaluator outputs for {len(llm_eval_results)} questions to {LLM_EVAL_PATH}")


=== [1/11] Evaluating question ===
  Raw LLM output (truncated):
  {   "question": "Which groups are considered at highest risk of severe influenza complications, and what medical or preventive interventions are recommended for them?",   "is_fully_answered": false,   "completeness_score": 4,   "grounding_score": 5,   "relevance_score": 5,   "missing_key_points": [     "Specific treatment options for high-risk groups",     "Detailed information on the effectivenes...
  Parsed JSON:
{
  "question": "Which groups are considered at highest risk of severe influenza complications, and what medical or preventive interventions are recommended for them?",
  "is_fully_answered": false,
  "completeness_score": 4,
  "grounding_score": 5,
  "relevance_score": 5,
  "missing_key_points": [
    "Specific treatment options for high-risk groups",
    "Detailed information on the effectiveness of antiviral medications"
  ],
  "high_level_rationale": "The provided chunks identify high-risk groups, such a

In [25]:
# ---------- COMPUTE AVERAGES OF METRICS ACROSS ALL QUESTIONS ----------

total_completeness = 0
total_grounding = 0
total_relevance = 0
count = 0

for q, result in llm_eval_results.items():
    # skip entries with errors
    if "completeness_score" not in result:
        continue

    total_completeness += result["completeness_score"]
    total_grounding     += result["grounding_score"]
    total_relevance     += result["relevance_score"]
    count += 1

if count > 0:
    avg_completeness = total_completeness / count
    avg_grounding    = total_grounding / count
    avg_relevance    = total_relevance / count

    safe_print("\n=== AVERAGED METRICS ACROSS ALL QUESTIONS ===")
    safe_print(f"Average completeness_score: {avg_completeness:.3f}")
    safe_print(f"Average grounding_score:    {avg_grounding:.3f}")
    safe_print(f"Average relevance_score:    {avg_relevance:.3f}")
else:
    safe_print("No valid LLM evaluation results found to average.")


=== AVERAGED METRICS ACROSS ALL QUESTIONS ===
Average completeness_score: 3.364
Average grounding_score:    4.455
Average relevance_score:    4.545
